In [1]:
# Install uproot if not already installed
!pip install uproot

# Install optuna if not already installed
!pip install optuna

# importing libraries
import numpy as np
import pandas as pd
import uproot
from glob import glob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.4/395.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 919.6/919.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 656.7/656.7 kB 20.7 MB/s eta 0:00:00


In [2]:
branches = ['fEvent', 'fX', 'fY', 'fZ', 'fEdep',
            'fdEdx', 'Ekin', 'TOF', 'TrackLength',
            'ScatteringAng', 'Momentum']

# cargamos los hits
def load_hits(pattern):
    files = sorted(glob(pattern))
    dfs = []
    for run_id, path in enumerate(files):
        with uproot.open(path) as f:
            tree = f['Hits']
            data = {b: tree[b].array(library='np') for b in branches}
            df = pd.DataFrame({k: v.tolist() for k, v in data.items()})
            df['run_id'] = run_id
            df['event_uid'] = df['run_id'].astype(str) + '_' + df['fEvent'].astype(str)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


hits_mu = load_hits('data/muon/output_run*.root')
hits_pi = load_hits('data/pion/output_run*.root')

print(hits_mu.shape)
print(hits_pi.shape)

ValueError: No objects to concatenate

In [ ]:
import os
from glob import glob

muon_files = glob('data/muon/output_run*.root')
pion_files = glob('data/pion/output_run*.root')

print(f"Found {len(muon_files)} files for 'data/muon/output_run*.root'")
if len(muon_files) == 0:
    print("No muon data files found. Please ensure they are uploaded to 'data/muon/' path.")
else:
    print(f"First muon file: {muon_files[0]}")

print(f"Found {len(pion_files)} files for 'data/pion/output_run*.root'")
if len(pion_files) == 0:
    print("No pion data files found. Please ensure they are uploaded to 'data/pion/' path.")
else:
    print(f"First pion file: {pion_files[0]}")

Found 0 files for 'data/muon/output_run*.root'
No muon data files found. Please ensure they are uploaded to 'data/muon/' path.
Found 0 files for 'data/pion/output_run*.root'
No pion data files found. Please ensure they are uploaded to 'data/pion/' path.


In [ ]:
def aggregate_events(hits_df, label):
    g = hits_df.groupby('event_uid')

    feats = pd.DataFrame(
        {
            "n_hits": g['fEvent'].count(),
            'n_unique_cells': g[['fX', 'fY', 'fZ']].apply(lambda x: len(x.drop_duplicates())),

            # Deposito de energia
            'edep_sum': g['fEdep'].sum(),
            'edep_max': g['fEdep'].max(),
            'edep_std': g['fEdep'].std().fillna(0),

            # dE/dx
            'dedx_mean': g['fdEdx'].mean(),
            'dedx_max': g['fdEdx'].max(),
            'dedx_std': g['fdEdx'].std().fillna(0),

            # energia cinetica
            'ekin_first': g['Ekin'].first(),
            'ekin_last': g['Ekin'].last(),
            'ekin_loss': g['Ekin'].first() - g['Ekin'].last(),

            # TOF: tiempo de vuelo
            'tof_first': g['TOF'].first(),
            'tof_last': g['TOF'].last(),
            'tof_range': g['TOF'].max() - g['TOF'].min(),

            # Longitud de trayectoria
            'track_first': g['TrackLength'].first(),
            'track_last': g['TrackLength'].last(),
            'track_mean': g['TrackLength'].mean(),

            # Scattering angle
            'scat_mean': g['ScatteringAng'].mean(),
            'scat_max': g['ScatteringAng'].max(),
            'scat_std': g['ScatteringAng'].std().fillna(0),

            # geometria transversal
            'radial_spread': g.apply(lambda x: np.sqrt(x['fX'].var() + x['fY'].var())).fillna(0),
            'z_span': g['fZ'].max() - g['fZ'].min(),
    })

    feats['label'] = label
    return feats.reset_index()

events_mu = aggregate_events(hits_mu, label=1)
events_pi = aggregate_events(hits_pi, label=0)

df = pd.concat([events_mu, events_pi], ignore_index=True)
print(df.shape)
print(df['label'].value_counts())





/tmp/ipykernel_184631/1304506705.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  'radial_spread': g.apply(lambda x: np.sqrt(x['fX'].var() + x['fY'].var())).fillna(0),


(39640, 24)
label
0    37686
1     1954
Name: count, dtype: int64


/tmp/ipykernel_184631/1304506705.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  'radial_spread': g.apply(lambda x: np.sqrt(x['fX'].var() + x['fY'].var())).fillna(0),


In [ ]:
df.head()

,event_uid,n_hits,n_unique_cells,edep_sum,edep_max,edep_std,dedx_mean,dedx_max,dedx_std,ekin_first,...,tof_range,track_first,track_last,track_mean,scat_mean,scat_max,scat_std,radial_spread,z_span,label
0,0_0,3,3,0.011298,0.010642,0.005957,0.000519,0.000570,0.000044,9.959221,...,0.010841,40.972700,60.001116,47.431299,0.002405,0.005716,0.002924,8.164966,0.0,1
1,0_1,23,2,0.014892,0.005200,0.001134,0.005381,0.039648,0.008341,9.937587,...,0.177779,53.601376,0.102756,7.939928,0.292034,1.173107,0.341510,4.869848,0.0,1
2,0_10,1,1,0.008106,0.008106,0.000000,0.000405,0.000405,0.000000,9.954342,...,0.000000,60.001189,60.001189,60.001189,0.005239,0.005239,0.000000,0.000000,0.0,1
3,0_100,5,1,0.016102,0.008229,0.003595,0.036192,0.177818,0.079173,9.956423,...,0.056992,40.621146,0.006062,29.514078,0.038343,0.187716,0.083505,0.000000,0.0,1
4,0_101,24,3,0.015754,0.007015,0.001541,0.004796,0.019874,0.005508,9.938223,...,0.113329,40.399218,0.632421,6.620003,0.279870,2.713721,0.553844,6.967835,0.0,1


In [ ]:
# Splitting the data into training and testing sets

from itertools import starmap
from sklearn.model_selection import train_test_split

FEATURES = [c for c in df.columns if c not in ('event_uid', 'label')]

X = df[FEATURES]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
stratify=y, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(31712, 22) (7928, 22) (31712,) (7928,)


In [ ]:
from xgboost import XGBClassifier

# Definición del modelo con los parámetros actualizados
model = XGBClassifier(
    n_estimators=300,  # Actualizado de 100 a 300
    max_depth=4,       # Actualizado de 5 a 4
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42
)

# Entrenamiento del modelo
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

NameError: name 'X_train' is not defined

In [ ]:
# Splitting the data into training and testing sets

from itertools import starmap
from sklearn.model_selection import train_test_split

FEATURES = [c for c in df.columns if c not in ('event_uid', 'label')]

X = df[FEATURES]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
stratify=y, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

NameError: name 'df' is not defined

In [ ]:
from xgboost import XGBClassifier

# Definición del modelo con los parámetros actualizados
model = XGBClassifier(
    n_estimators=300,  # Actualizado de 100 a 300
    max_depth=4,       # Actualizado de 5 a 4
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42
)

# Entrenamiento del modelo
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

NameError: name 'X_train' is not defined

In [ ]:
print('Shape of X_train:', X_train.shape)
print('Shape of X_test:', X_test.shape)

NameError: name 'X_train' is not defined

In [2]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay, ConfusionMatrixDisplay
import shap

# 1. Predicciones y Reporte de texto
y_pred, y_prob = model.predict(X_test), model.predict_proba(X_test)[:, 1]
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}\n")
print(classification_report(y_test, y_pred))

# 2. Gráficas (ROC, Matriz de Confusión y Curva de Aprendizaje)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax[0])
ax[0].set_title("Curva ROC")

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues', colorbar=False, ax=ax[1])
ax[1].set_title("Matriz de Confusión")

ax[2].plot(model.evals_result()['validation_0']['logloss'], color='red')
ax[2].set_title("Curva de Aprendizaje (Log Loss)")

plt.tight_layout()
plt.show()

# 3. Feature Importance con SHAP
shap.summary_plot(shap.TreeExplainer(model).shap_values(X_test), X_test, plot_type="dot")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Juntar la energía, el valor real y la predicción en un pequeño DataFrame
df_analisis = pd.DataFrame({
    'Energia': X_test['ekin_first'],  # Cambia a 'edep_sum' si prefieres esa energía
    'Fallo': y_test != y_pred         # True si el modelo se equivocó, False si acertó
})

# 2. Crear 10 bins (rangos) de energía usando qcut (para que tengan una cantidad similar de datos)
# Si prefieres rangos de tamaño exacto matemáticamente, cambia qcut por cut
df_analisis['Bin_Energia'] = pd.qcut(df_analisis['Energia'], q=10, duplicates='drop')

# 3. Calcular la tasa de fallo por cada bin
resumen = df_analisis.groupby('Bin_Energia')['Fallo'].agg(['mean', 'count']).reset_index()
resumen['Tasa_Fallo_%'] = resumen['mean'] * 100
resumen['Bin_Str'] = resumen['Bin_Energia'].astype(str)

# 4. Graficar los resultados
plt.figure(figsize=(10, 5))
barras = plt.bar(resumen['Bin_Str'], resumen['Tasa_Fallo_%'], color='salmon', edgecolor='black')

plt.title('Análisis de Errores por Rango de Energía (ekin_first)')
plt.xlabel('Intervalos de Energía')
plt.ylabel('Tasa de Error (%)')
plt.xticks(rotation=45, ha='right')

# Añadir el número de eventos (n) en la parte superior de cada barra
for i, barra in enumerate(barras):
    yval = barra.get_height()
    plt.text(barra.get_x() + barra.get_width()/2, yval + 0.5,
             f"n={resumen['count'].iloc[i]}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def objective(trial):
    # 1. Definir el "espacio de búsqueda" (los rangos donde Optuna va a buscar)
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'random_state': 42,
        'eval_metric': 'logloss'
    }

    # 2. Crear el modelo con los parámetros sugeridos en este intento
    modelo_trial = XGBClassifier(**param)

    # 3. Entrenar el modelo (apagamos verbose para no llenar la pantalla de texto)
    modelo_trial.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # 4. Hacer predicciones y calcular el ROC-AUC
    preds_proba = modelo_trial.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, preds_proba)

    return auc # Optuna usará este valor para saber si va por buen camino

# Ejecución del Estudio
print("Iniciando la búsqueda de hiperparámetros con Optuna...")
# direction='maximize' porque un ROC-AUC más alto es mejor
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20) # Puedes subir n_trials a 50 o 100 si tienes tiempo

# Resultados
print("\n" + "="*40)
print(f"Búsqueda terminada")
print(f"Mejor ROC-AUC alcanzado: {study.best_value:.4f}")
print("Mejores Hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"   - {key}: {value}")
print("="*40)

In [ ]:
#Clase MuonPionClassifier

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay, ConfusionMatrixDisplay
from xgboost import XGBClassifier
import optuna
import shap
import warnings
warnings.filterwarnings('ignore')

class MuonPionClassifier:
    """
    Clase para manejar todo el pipeline de clasificación entre Muones y Piones:
    Optimización, entrenamiento, evaluación y análisis de errores.
    """
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.model = None
        self.best_params = None

    def optimize(self, X_train, y_train, X_test, y_test, n_trials=20):
        """Busca los mejores hiperparámetros usando Optuna."""
        print(f"--- Iniciando optimización con Optuna ({n_trials} intentos) ---")

        def objective(trial):
            param = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
                'max_depth': trial.suggest_int('max_depth', 3, 9),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
                'random_state': self.random_state,
                'eval_metric': 'logloss'
            }

            modelo_trial = XGBClassifier(**param)
            modelo_trial.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
            preds_proba = modelo_trial.predict_proba(X_test)[:, 1]
            return roc_auc_score(y_test, preds_proba)

        study = optuna.create_study(direction='maximize')
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study.optimize(objective, n_trials=n_trials)

        self.best_params = study.best_params
        print(f"Optimización terminada. Mejor ROC-AUC: {study.best_value:.4f}")
        return self.best_params

    def train(self, X_train, y_train, X_test, y_test):
        """Entrena el modelo final con los mejores parámetros encontrados."""
        if self.best_params is None:
            print("Aviso: Entrenando con parámetros por defecto. Usa optimize() primero para mejores resultados.")
            params = {'random_state': self.random_state, 'eval_metric': 'logloss'}
        else:
            params = self.best_params.copy()
            params['random_state'] = self.random_state
            params['eval_metric'] = 'logloss'

        print("--- Entrenando modelo final ---")
        self.model = XGBClassifier(**params)
        self.model.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_test, y_test)],
            verbose=False # Se puede camviar a 50 para ver el progreso de los árboles
        )
        print("Entrenamiento completo")

    def evaluate(self, X_test, y_test):
        """Genera el reporte de clasificación, curva ROC, Matriz de Confusión y Curvas de Aprendizaje."""
        if self.model is None:
            raise ValueError("El modelo no ha sido entrenado. Llama a train() primero.")

        y_pred = self.model.predict(X_test)
        y_prob = self.model.predict_proba(X_test)[:, 1]

        print("\n" + "="*50)
        print(f"ROC-AUC SCORE: {roc_auc_score(y_test, y_prob):.4f}")
        print("="*50)
        print(classification_report(y_test, y_pred))

        fig, ax = plt.subplots(1, 3, figsize=(16, 4))

        RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax[0])
        ax[0].set_title("Curva ROC")

        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues', colorbar=False, ax=ax[1])
        ax[1].set_title("Matriz de Confusión")

        results = self.model.evals_result()
        ax[2].plot(results['validation_0']['logloss'], label='Train Loss')
        ax[2].plot(results['validation_1']['logloss'], label='Test Loss', color='red')
        ax[2].set_title("Curva de Aprendizaje")
        ax[2].set_xlabel("Árboles (Épocas)")
        ax[2].set_ylabel("Log Loss")
        ax[2].legend()

        plt.tight_layout()
        plt.show()

    def plot_shap(self, X_test):
        """Calcula y grafica la importancia de las variables usando SHAP."""
        print("\n--- Analizando impacto de variables (SHAP) ---")
        explainer = shap.TreeExplainer(self.model)
        shap_values = explainer.shap_values(X_test)
        shap.summary_plot(shap_values, X_test, plot_type="dot")

    def analyze_errors_by_energy(self, X_test, y_test, energy_col='ekin_first'):
        """Analiza la tasa de fallos separando los datos en rangos de energía."""
        y_pred = self.model.predict(X_test)

        df_analisis = pd.DataFrame({
            'Energia': X_test[energy_col],
            'Fallo': y_test != y_pred
        })

        df_analisis['Bin_Energia'] = pd.qcut(df_analisis['Energia'], q=10, duplicates='drop')
        resumen = df_analisis.groupby('Bin_Energia')['Fallo'].agg(['mean', 'count']).reset_index()
        resumen['Tasa_Fallo_%'] = resumen['mean'] * 100
        resumen['Bin_Str'] = resumen['Bin_Energia'].astype(str)

        plt.figure(figsize=(10, 4))
        barras = plt.bar(resumen['Bin_Str'], resumen['Tasa_Fallo_%'], color='salmon', edgecolor='black')

        plt.title(f'Análisis de Errores por Rango de Energía ({energy_col})')
        plt.xlabel('Intervalos de Energía')
        plt.ylabel('Tasa de Error (%)')
        plt.xticks(rotation=45, ha='right')

        for i, barra in enumerate(barras):
            yval = barra.get_height()
            plt.text(barra.get_x() + barra.get_width()/2, yval + 0.5,
                     f"n={resumen['count'].iloc[i]}", ha='center', va='bottom', fontsize=8)

        plt.tight_layout()
        plt.show()

# CÓMO USAR LA CLASE (Ejecución del pipeline)

# 1. Instanciamos el clasificador
clasificador = MuonPionClassifier()

# 2. Buscamos los mejores hiperparámetros (se puede subir n_trials a 50)
clasificador.optimize(X_train, y_train, X_test, y_test, n_trials=20)

# 3. Entrenamos el modelo con los parámetros encontrados
clasificador.train(X_train, y_train, X_test, y_test)

# 4. Evaluamos métricas y gráficas generales
clasificador.evaluate(X_test, y_test)

# 5. Vemos la importancia de las variables con SHAP
clasificador.plot_shap(X_test)

# 6. Analizamos los fallos por energía
clasificador.analyze_errors_by_energy(X_test, y_test, energy_col='ekin_first')